In [1]:
import pandas as pd
import numpy as np
import warnings
from Univariate import Univariate
warnings.filterwarnings("ignore")

dataset = pd.read_csv("kidney_disease.csv")

qual = ['sg','al','su','rbc','pc','pcc','ba','htn','dm','cad','appet','pe','ane','classification']
quan = ['age','bp','bgr','bu','sc','sod','pot','hemo','pcv','wc','rc']

# pre-processing categorical columns

In [2]:
categorical_columns = dataset[qual]

# Count of missing values
null_counts = categorical_columns.isnull().sum()

# Percentage of missing values
null_percentage = (null_counts / len(dataset)) * 100

# Combine into one DataFrame for analysis
missing_summary = pd.DataFrame({
    "Missing Count": null_counts,
    "Missing %": null_percentage.round(2)
})

print(missing_summary)

                Missing Count  Missing %
sg                         36      12.86
al                         35      12.50
su                         38      13.57
rbc                       107      38.21
pc                         50      17.86
pcc                         4       1.43
ba                          4       1.43
htn                         1       0.36
dm                          1       0.36
cad                         1       0.36
appet                       0       0.00
pe                          0       0.00
ane                         0       0.00
classification              0       0.00


#### For rbc the missing is high, so instead of directly replacing by its mode, trying to replace it proportionatly based on "ckd/notckd", i.e, for rbc missing columns with ckd as classification replace it with mode of rbc with ckd and rbc missing columns with notckd  replace it with mode of rbc with notckd

#### checking rbc frequency for "ckd" data

In [3]:
 
dataset.loc[dataset["classification"] == "ckd", ['rbc', "classification"]].value_counts()


rbc       classification
normal    ckd               43
abnormal  ckd               29
Name: count, dtype: int64

#### checking rbc frequency for "notckd" data

In [4]:
# checking rbc frequency for "notckd" data
dataset.loc[dataset["classification"] == "notckd", ["rbc", "classification"]].value_counts()


rbc     classification
normal  notckd            101
Name: count, dtype: int64

#### For both ckd and notckd mode is normal, so replacing Nan with "normal", but there is risk of mode getting biased towards rbc=normal, because only 29 rows will be abnormal among 280 rows. First try with this approach and evaluate the model. Advanced approach --> KNN imputer

In [5]:
dataset['rbc'].fillna("normal",inplace=True)

dataset['rbc'].isna().sum()

np.int64(0)

In [6]:
#did the same for 'pc' column, Normal is the mode
dataset['pc'].fillna("normal",inplace=True)

dataset['pc'].isna().sum()

np.int64(0)

In [7]:
dataset.loc[dataset["classification"] == "ckd", ['sg', "classification"]].value_counts()
#sg for ckd mode is 61

sg     classification
1.010  ckd               61
1.015  ckd               48
1.020  ckd               23
1.025  ckd                5
1.005  ckd                4
Name: count, dtype: int64

In [8]:
dataset.loc[dataset["classification"] == "notckd", ['sg', "classification"]].value_counts()
#sg for notckd mode is 52

sg     classification
1.025  notckd            52
1.020  notckd            51
Name: count, dtype: int64

In [9]:
# Prortionaltely filling Nan for "sg" , mode of sg with ckd, mode of sg with not ckd

dataset['sg'] = dataset.groupby('classification')['sg'].transform(
    lambda x: x.fillna(x.mode()[0])
)

In [10]:
dataset['sg'].isna().sum()

np.int64(0)

In [11]:
# doing the same for 'al', 'su'

dataset['al'] = dataset.groupby('classification')['al'].transform(
    lambda x: x.fillna(x.mode()[0])
)

dataset['su'] = dataset.groupby('classification')['su'].transform(
    lambda x: x.fillna(x.mode()[0])
)

In [12]:
dataset[qual].isna().sum()

sg                0
al                0
su                0
rbc               0
pc                0
pcc               4
ba                4
htn               1
dm                1
cad               1
appet             0
pe                0
ane               0
classification    0
dtype: int64

In [16]:
# all mojor .columns are replaced, pcc, ba, htn, dm, cad have very few so dropping them

dataset.dropna(subset=qual, inplace=True)

dataset[qual].isna().sum()

sg                0
al                0
su                0
rbc               0
pc                0
pcc               0
ba                0
htn               0
dm                0
cad               0
appet             0
pe                0
ane               0
classification    0
dtype: int64

In [19]:
dataset.shape

(275, 26)

# pre-processing numerical columns

In [20]:
dataset[quan].isna().sum()

age      5
bp       8
bgr     33
bu      14
sc      12
sod     67
pot     68
hemo    38
pcv     51
wc      78
rc      94
dtype: int64

### using imputer to replace numerical columns with mean

In [26]:
from sklearn.impute import SimpleImputer

# Numerical: mean imputation
imputer_num = SimpleImputer(strategy='mean')
dataset[quan] = imputer_num.fit_transform(dataset[quan])

dataset[quan] = dataset[quan].round(1)

In [27]:
dataset.isna().sum()

id                0
age               0
bp                0
sg                0
al                0
su                0
rbc               0
pc                0
pcc               0
ba                0
bgr               0
bu                0
sc                0
sod               0
pot               0
hemo              0
pcv               0
wc                0
rc                0
htn               0
dm                0
cad               0
appet             0
pe                0
ane               0
classification    0
dtype: int64

In [29]:
# Export dataset to CSV
dataset.to_csv("cleaned_ckd_dataset.csv", index=False)